In [ ]:
# Install required packages
!pip install ultralytics -q
!pip install roboflow -q

import os
import json
import cv2
import shutil
import yaml
import random
from pathlib import Path
from tqdm import tqdm
from google.colab import drive
from IPython.display import Image, display

print("All packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 113.9 MB/s eta 0:00:00
All packages installed successfully!


In [ ]:

# Mount Google Drive
drive.mount('/content/drive')

print("\nGoogle Drive mounted!")

Mounted at /content/drive

Google Drive mounted!


In [ ]:

DATASET_ROOT = "/content/drive/MyDrive/Gun_Action_Recognition_Dataset"

OUTPUT_ROOT = "/content/yolo_dataset" 

FRAME_SKIP = 5
FRAME_SKIP_NO_GUN = 50  # extract fewer background frames
TRAIN_SPLIT = 0.7  # changes to 70 , 30
EPOCHS = 100
IMG_SIZE = 640
BATCH_SIZE = 8    # changed batch size to 8
MODEL_SIZE = 'yolov8s'

print("Configuration set!")
print(f"\nDataset: {DATASET_ROOT}")
print(f"Output: {OUTPUT_ROOT}")
print(f"Model: {MODEL_SIZE}")
print(f"Epochs: {EPOCHS}")

Configuration set!

Dataset: /content/drive/MyDrive/Gun_Action_Recognition_Dataset
Output: /content/yolo_dataset
Model: yolov8n
Epochs: 35


In [ ]:
def extract_and_convert(dataset_root, output_root, frame_skip=5, train_split=0.8):
    """Extract frames and convert labels simultaneously to minimize background images"""
    import os
    import json
    import cv2
    import random
    from pathlib import Path
    from tqdm import tqdm

    print("=" * 60)
    print("EXTRACTING FRAMES AND CONVERTING LABELS")
    print("=" * 60)

    output_root = Path(output_root)
    dataset_root = Path(dataset_root)

    # Create directories
    for split in ['train', 'val']:
        (output_root / split / 'images').mkdir(parents=True, exist_ok=True)
        (output_root / split / 'labels').mkdir(parents=True, exist_ok=True)

    categories = ['Handgun', 'Machine_Gun', 'No_Gun']
    class_mapping = {'Handgun': 0, 'Machine_Gun': 1}

    # Collect all video folders
    all_videos = []
    for category in categories:
        category_path = dataset_root / category
        if category_path.exists():
            video_folders = [f for f in category_path.iterdir() if f.is_dir()]
            all_videos.extend([(category, vf) for vf in video_folders])

    # Shuffle and split
    random.shuffle(all_videos)
    split_idx = int(len(all_videos) * train_split)
    train_videos = all_videos[:split_idx]
    val_videos = all_videos[split_idx:]

    print(f"\nDataset Split:")
    print(f"   Training: {len(train_videos)} videos")
    print(f"   Validation: {len(val_videos)} videos\n")

    total_images = 0
    total_annotations = 0
    total_backgrounds = 0

    for split_name, video_list in [('train', train_videos), ('val', val_videos)]:
        print(f"\n{'='*60}")
        print(f"Processing {split_name.upper()} videos...")
        print(f"{'='*60}\n")

        for category, video_folder in tqdm(video_list, desc=f"{split_name}"):
            video_path = video_folder / 'video.mp4'
            if not video_path.exists():
                continue

            # Load labels if any
            label_json = video_folder / 'label.json'
            label_data = {}
            if label_json.exists():
                with open(label_json, 'r') as f:
                    label_data = json.load(f)

            # Build a lookup for annotations by frame number (1-indexed in json)
            annotations_by_frame = {}
            if 'annotations' in label_data:
                for ann in label_data['annotations']:
                    img_id = ann.get('image_id', 0)
                    if img_id not in annotations_by_frame:
                        annotations_by_frame[img_id] = []
                    annotations_by_frame[img_id].append(ann)

            cap = cv2.VideoCapture(str(video_path))
            
            frame_idx = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                current_skip = 30 if category == 'No_Gun' else frame_skip

                if frame_idx % current_skip == 0:
                    json_image_id = frame_idx + 1
                    
                    has_boxes = False
                    yolo_lines = []
                    
                    if json_image_id in annotations_by_frame:
                        height, width = frame.shape[:2]
                        for ann in annotations_by_frame[json_image_id]:
                            bbox = ann.get('bbox', [0, 0, 0, 0])
                            if len(bbox) == 4:
                                x, y, w, h = bbox
                                if w > 0 and h > 0:
                                    x_center = max(0, min(1, (x + w/2) / width))
                                    y_center = max(0, min(1, (y + h/2) / height))
                                    norm_w = max(0, min(1, w / width))
                                    norm_h = max(0, min(1, h / height))
                                    class_id = class_mapping.get(category, 0)
                                    yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")
                                    has_boxes = True
                    
                    should_save = False
                    if has_boxes:
                        should_save = True
                        total_annotations += len(yolo_lines)
                    elif category == 'No_Gun':
                        should_save = True
                        total_backgrounds += 1
                    else:
                        if random.random() < 0.05:
                            should_save = True
                            total_backgrounds += 1
                            
                    if should_save:
                        frame_name = f"{category}_{video_folder.name}_f{frame_idx:05d}"
                        img_path = output_root / split_name / 'images' / f"{frame_name}.jpg"
                        label_path = output_root / split_name / 'labels' / f"{frame_name}.txt"
                        
                        cv2.imwrite(str(img_path), frame)
                        with open(label_path, 'w') as f:
                            for line in yolo_lines:
                                f.write(line)
                        total_images += 1

                frame_idx += 1
            cap.release()

    print(f"\nEXTRACTION COMPLETE!")
    print(f"Total Saved Images: {total_images}")
    print(f"Total Bounding Boxes: {total_annotations}")
    if total_images > 0:
        print(f"Total Background Images: {total_backgrounds} ({(total_backgrounds/total_images*100):.1f}% of dataset)")

# Run combined extraction and conversion
extract_and_convert(DATASET_ROOT, OUTPUT_ROOT, FRAME_SKIP, TRAIN_SPLIT)


EXTRACTING FRAMES FROM VIDEOS

Dataset Split:
   Training: 318 videos
   Validation: 80 videos
   Frame skip: Every 5 frames


Processing TRAIN videos...



train: 100%|██████████| 318/318 [10:19<00:00,  1.95s/it]



✓ Extracted 15918 frames for train

Processing VAL videos...



val: 100%|██████████| 80/80 [02:46<00:00,  2.08s/it]


✓ Extracted 4025 frames for val

FRAME EXTRACTION COMPLETE!


True

In [ ]:
# Label conversion is now unified with extraction above.

CONVERTING LABELS TO YOLO FORMAT

📋 Class Mapping:
   0: Handgun
   1: Machine_Gun

Processing TRAIN labels...



train: 100%|██████████| 15918/15918 [04:03<00:00, 65.31it/s] 



✓ Converted 6677 annotations for train

Processing VAL labels...



val: 100%|██████████| 4025/4025 [01:08<00:00, 59.13it/s] 


✓ Converted 1870 annotations for val

LABEL CONVERSION COMPLETE!
   Total annotations: 8547


True

In [ ]:
def create_yaml(output_root):

    output_root = Path(output_root)

    config = {
        'path': str(output_root.absolute()),
        'train': 'train/images',
        'val': 'val/images',
        'nc': 2,
        'names': {
            0: 'Handgun',
            1: 'Machine_Gun'
        }
    }

    yaml_path = output_root / 'gun_detection.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

    print("=" * 60)
    print("YAML CONFIGURATION CREATED")
    print("=" * 60)
    print(f"\nLocation: {yaml_path}\n")
    print("Contents:")
    print("-" * 40)
    with open(yaml_path, 'r') as f:
        print(f.read())
    print("-" * 40)

    return yaml_path

# Create YAML
yaml_path = create_yaml(OUTPUT_ROOT)
print("\nConfiguration ready for training!")

YAML CONFIGURATION CREATED

Location: /content/yolo_dataset/gun_detection.yaml

Contents:
----------------------------------------
names:
  0: Handgun
  1: Machine_Gun
nc: 2
path: /content/yolo_dataset
train: train/images
val: val/images

----------------------------------------

Configuration ready for training!


In [ ]:
from ultralytics import YOLO
import os

checkpoint_dir = '/content/drive/MyDrive/CrimeGuard_AI_Checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

print("=" * 60)
print("STARTING YOLO TRAINING")
print("=" * 60)
print(f"\nTraining Configuration:")
print(f"   Model: {MODEL_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Image Size: {IMG_SIZE}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Config: {yaml_path}")
print(f"   Checkpoint Dir: {checkpoint_dir}\n")

last_checkpoint = f'{checkpoint_dir}/gun_detection/weights/last.pt'

if os.path.exists(last_checkpoint):
    print(f"FOUND EXISTING CHECKPOINT!")
    print(f"   Resuming from: {last_checkpoint}\n")
    model = YOLO(last_checkpoint)

    # Resume training
    results = model.train(resume=True)

else:
    print("Starting fresh training...\n")

    # Load pretrained model
    model = YOLO(f'{MODEL_SIZE}.pt')

    print("Starting training...\n")
    print("=" * 60)

    # Train the model
    results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=20,  # Early stopping,
    project=checkpoint_dir,
    name='gun_detection',
    save_period=5,
    device=0
)

print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)
print(f"\nBest model saved to Google Drive:")
print(f"   {checkpoint_dir}/gun_detection/weights/best.pt")
print(f"\nResults folder:")
print(f"   {checkpoint_dir}/gun_detection/")


final_models_dir = '/content/drive/MyDrive/CrimeGuard_AI_Models'
os.makedirs(final_models_dir, exist_ok=True)

import shutil
best_model = f'{checkpoint_dir}/gun_detection/weights/best.pt'
last_model = f'{checkpoint_dir}/gun_detection/weights/last.pt'

if os.path.exists(best_model):
    shutil.copy(best_model, f'{final_models_dir}/gun_detection_best.pt')
    print(f"\nFINAL MODEL COPIED TO:")
    print(f"   {final_models_dir}/gun_detection_best.pt")
    print(f"\nDownload from Google Drive: CrimeGuard_AI_Models folder")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 STARTING YOLO TRAINING

📋 Training Configuration:
   Model: yolov8n
   Epochs: 35
   Image Size: 640
   Batch Size: 16
   Config: /content/yolo_dataset/gun_detection.yaml

💾 IMPORTANT: Saving to Google Drive!
   Checkpoint Dir: /content/drive/MyDrive/CrimeGuard_AI_Checkpoints

✅ FOUND EXISTING CHECKPOINT!
   Resuming from: /content/drive/MyDrive/CrimeGuard_AI_Checkpoints/gun_detection/weights/last.pt

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=Fa

## 📊 STEP 9: View Training Results

Let's look at the training metrics and performance

In [ ]:
from IPython.display import Image as IPImage

print("=" * 60)
print("📊 TRAINING RESULTS")
print("=" * 60)

results_dir = Path('/content/runs/detect/gun_detection')

# Display training curves
print("\nTraining Curves:\n")
results_img = results_dir / 'results.png'
if results_img.exists():
    display(IPImage(filename=str(results_img), width=900))

# Display confusion matrix
print("\nConfusion Matrix:\n")
confusion_img = results_dir / 'confusion_matrix.png'
if confusion_img.exists():
    display(IPImage(filename=str(confusion_img), width=600))

# Display sample predictions
print("\nSample Predictions:\n")
val_batch = results_dir / 'val_batch0_pred.jpg'
if val_batch.exists():
    display(IPImage(filename=str(val_batch), width=900))

📊 TRAINING RESULTS

📈 Training Curves:


🎯 Confusion Matrix:


🔍 Sample Predictions:



## ✅ STEP 10: Validate Model

Run validation on the validation set to get final metrics

In [ ]:
print("=" * 60)
print("✅ VALIDATING MODEL")
print("=" * 60)

# Load best model
best_model = YOLO('/content/drive/MyDrive/CrimeGuard_AI_Models/gun_detection_best.pt')

# Run validation
metrics = best_model.val()

print("\nValidation Metrics:")
print(f"\n   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   Precision: {metrics.box.mp:.4f}")
print(f"   Recall: {metrics.box.mr:.4f}")

# Per-class metrics
print("\n   Per-Class AP50:")
class_names = ['Handgun', 'Machine_Gun']
for i, name in enumerate(class_names):
    print(f"      {name}: {metrics.box.ap50[i]:.4f}")

✅ VALIDATING MODEL
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2696.4±1040.4 MB/s, size: 249.9 KB)
val: Scanning /content/yolo_dataset/val/labels.cache... 3978 images, 2257 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3978/3978 1.5Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 249/249 5.6it/s 44.3s
                   all       3978       1721      0.963      0.947      0.979      0.701
               Handgun        899        899       0.97      0.953      0.983      0.663
           Machine_Gun        822        822      0.957      0.942      0.976       0.74
Speed: 1.1ms preprocess, 2.8ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /content/runs/detect/val

📊 Validation Metrics:

   mAP50: 0.9794
   mAP50-95: 0.7014
   Precisi

## 🎯 STEP 11: Test on Sample Video

Test the trained model on a sample video from your dataset

In [ ]:
dataset_path = Path(DATASET_ROOT)
sample_video = '/content/video.mp4'

if sample_video:
    print(f"🎬 Testing on: {Path(sample_video).name}\n")

    # Run inference
    results = best_model.predict(
        source=sample_video,
        save=True,
        conf=0.25,
        project='/content/test_results',
        name='gun_detection',
        exist_ok=True
    )

    print("\nTest complete!")
    print("Results saved in: /content/test_results/gun_detection/")
else:
    print("No sample video found for testing")

🎬 Testing on: video.mp4


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/175) /content/video.mp4: 480x640 (no detections), 156.7ms
video 1/1 (frame 2/175) /content/video.mp4: 480x640 (no detections), 27.4ms
video 1/1 (frame 3/175) /content/video.mp4: 480x640 (no detections), 23.6ms
video 1/1 (frame 4/175) /content/video.mp4: 480x640 (no detections), 22.4ms
video 1/1 (frame 5/175) /content/video.mp4: 480x640 (no detections), 19.9ms
video 1/1 (frame 6/175) /content/video.mp4: 480x640 